# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library. All record sets and fields are referenced by their `@id`. The workflow includes data loading, overview, extraction, exploratory analysis, and basic visualization steps.

### Dataset Source
The dataset source is a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all record sets, their `@id`s, associated fields, and each field's `@id`. This helps in selecting which tables to analyze later.

In [ ]:
from pprint import pprint
print("Record Sets (by @id):")

# Enumerate all record sets and fields by their @id
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- Record Set @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields/Columns (by @id):")
    # List all fields in this record set
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, name: {field.name}")
    elif hasattr(rs, 'columns'):
        for col in rs.columns:
            print(f"    - Column @id: {col.id}, name: {col.name}")
    print()

## 3. Data Extraction
We extract data for one or more record sets by their `@id`. The resulting DataFrames use fields as columns, referenced by their `@id` for clarity and reproducibility.

> *Replace the record set and field IDs below as appropriate for your analysis after reviewing the record sets above.*

In [ ]:
# Define the record set(s) to load by @id
# (Example: check printout above. Replace with real IDs as per your dataset)
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record_set @id={record_set_id}, shape: {df.shape}")

# Preview columns of the main data table (first record set listed)
main_rs_id = record_set_ids[0]
print(f"Columns for main record set (@id={main_rs_id}):")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering and normalization to numeric fields and group by a categorical field using their `@id` references.

Example workflow: remove records where a selected numeric field is below a threshold, normalize the column, and group by a key attribute.

In [ ]:
# Choose numeric field and group field by their @id
# Replace with IDs found in the previous overview cell, e.g. 'cr:field:abundance', 'cr:field:accession'
numeric_field_id = dataframes[main_rs_id].select_dtypes('number').columns[0] if len(dataframes[main_rs_id].select_dtypes('number').columns) > 0 else dataframes[main_rs_id].columns[0]
group_field_id = dataframes[main_rs_id].columns[-1]  # Example: last column

threshold = dataframes[main_rs_id][numeric_field_id].mean() if pd.api.types.is_numeric_dtype(dataframes[main_rs_id][numeric_field_id]) else 0
filtered_df = dataframes[main_rs_id]
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by group_field_id if it's in the DataFrame
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Column {numeric_field_id} is not numeric. Please choose a numeric field.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relation to the group field, where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    plt.figure(figsize=(8, 4))
    plt.title(f"Distribution of {numeric_field_id}")
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.show()

    if group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Using the `mlcroissant` library, we loaded the FAIR^2 dataset, explored its structure by `@id`, extracted key record sets, and performed basic exploratory analysis and plotting. For deeper analyses, choose additional fields of interest by `@id` and apply statistical or domain-specific processing as needed.